# 🔍 Debug: Why Maximum Steps Reached?

This notebook helps identify why training is hitting the max_steps=500 limit.

We'll:
1. Load and inspect a training epoch
2. Calculate theoretical step requirements
3. Debug decoder steps one-by-one
4. Monitor constraint violations
5. Find where agents get stuck
6. Pinpoint the bottleneck

## 1️⃣ Import & Setup

In [1]:
%load_ext autoreload
%autoreload 2

import os
import torch
import numpy as np
from rl4co.data.utils import load_npz_to_tensordict

from parco.envs.pvrpwdp import PVRPWDPVEnv
from parco.models import PARCOPolicy

# Setup
project_root = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'examples' else os.getcwd()
os.chdir(project_root)
print(f"✅ Working directory: {os.getcwd()}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Device: {device}")

e:\Đồ án\parco\.venv\Lib\site-packages\torchrl\data\replay_buffers\samplers.py:37: UserWarning: Failed to import torchrl C++ binaries. Some modules (eg, prioritized replay buffers) may not work with your installation. This is likely due to a discrepancy between your package version and the PyTorch version. Make sure both are compatible. Usually, torchrl majors follow the pytorch majors within a few days around the release. For instance, TorchRL 0.5 requires PyTorch 2.4.0, and TorchRL 0.6 requires PyTorch 2.5.0.
  warnings.warn(EXTENSION_WARNING)
e:\Đồ án\parco\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
e:\Đồ án\parco\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .au

✅ Working directory: e:\Đồ án\parco
✅ Device: cuda


## 2️⃣ Load & Inspect Training Data

In [2]:
print("="*80)
print("📦 LOADING TRAINING DATA")
print("="*80)

# Load training epoch
td_loaded = load_npz_to_tensordict('data/train_data_npz/pvrpwdp_epoch_00.npz')
td_batch = td_loaded[:4]  # Small batch for debugging

print(f"\n📊 Batch Size: {td_batch.batch_size}")
print(f"\n🗺️  Locations:")
print(f"   Shape: {td_batch['locs'].shape}  # [B, N, 2]")
print(f"   Depot: {td_batch['depot'][0]}")
print(f"   Customers: {td_batch['locs'].shape[1]} locations")

print(f"\n🚗 Agents:")
print(f"   Count: {td_batch['agents_speed'].shape[1]}")
print(f"   Speed: {td_batch['agents_speed'][0]}")
print(f"   Capacity: {td_batch['agents_capacity'][0]}")
print(f"   Endurance: {td_batch['agents_endurance'][0]}")

print(f"\n📦 Demands:")
print(f"   Shape: {td_batch['demand'].shape}")
print(f"   Values[0]: {td_batch['demand'][0]}")

print(f"\n⏰ Time Windows:")
print(f"   Shape: {td_batch['time_windows'].shape}")
print(f"   First 3 [0]: {td_batch['time_windows'][0, :3]}")

print(f"\n🍎 Waiting Time (Freshness):")
print(f"   Shape: {td_batch['waiting_time'].shape}")
print(f"   Values[0]: {td_batch['waiting_time'][0]}")

# Summary
num_customers = td_batch['locs'].shape[1]
num_agents = td_batch['agents_speed'].shape[1]
print(f"\n📋 SUMMARY:")
print(f"   Customers: {num_customers}")
print(f"   Agents: {num_agents}")
print(f"   Total nodes (depot copies + customers): {num_agents + num_customers}")

📦 LOADING TRAINING DATA

📊 Batch Size: torch.Size([4])

🗺️  Locations:
   Shape: torch.Size([4, 20, 2])  # [B, N, 2]
   Depot: tensor([0., 0.])
   Customers: 20 locations

🚗 Agents:
   Count: 4
   Speed: tensor([15.6464, 15.6464, 31.2928, 31.2928])
   Capacity: tensor([200.0000, 200.0000,   2.2700,   2.2700])
   Endurance: tensor([4009.3093, 4009.3093,  700.0000,  700.0000])

📦 Demands:
   Shape: torch.Size([4, 20])
   Values[0]: tensor([ 0.6369,  0.5610, 30.3189, 36.5100,  0.5564, 39.3464,  0.4213,  0.1506,
         0.3132, 45.4155,  0.6672, 48.9188,  0.1506,  0.5523,  1.0752,  0.9865,
         0.4744,  0.2766,  0.2236,  0.2977])

⏰ Time Windows:
   Shape: torch.Size([4, 20, 2])
   First 3 [0]: tensor([[1773., 3176.],
        [ 130., 1477.],
        [   0., 1014.]])

🍎 Waiting Time (Freshness):
   Shape: torch.Size([4, 20])
   Values[0]: tensor([3600., 3600., 3600., 3600., 3600., 3600., 3600., 3600., 3600., 3600.,
        3600., 3600., 3600., 3600., 3600., 3600., 3600., 3600., 3600., 

## 3️⃣ Analyze Episode Length Requirements

In [3]:
print("="*80)
print("📊 THEORETICAL STEP REQUIREMENTS")
print("="*80)

# Environment setup
env = PVRPWDPVEnv(
    epoch_data_dir=os.path.join(project_root, "data", "train_data_npz"),
    epoch_file_pattern="pvrpwdp_epoch_{epoch:02d}.npz",
    use_epoch_data=True,
    fallback_to_generator=False,
)

# Reset
td_reset = env.reset(td_batch.clone())

print(f"\n📐 Calculation:")
print(f"   Customers to visit: {num_customers}")
print(f"   Agents available: {num_agents}")
print(f"   Optimal distribution: {num_customers / num_agents:.1f} customers per agent")

print(f"\n🔢 Step Estimates:")
print(f"   Best case (balanced): {num_customers + num_agents} steps")
print(f"     = {num_customers} customer visits + {num_agents} returns to depot")

print(f"\n   Realistic case (unbalanced): {num_customers + num_agents * 2} steps")
print(f"     = {num_customers} customer visits + {num_agents * 2} intermediate depot returns")

print(f"\n   Worst case (all back to depot between customers): {num_customers * 2} steps")
print(f"     = each customer requires go + return = {num_customers * 2}")

print(f"\n⚠️  Environment Limit:")
print(f"   max_steps = 500 (set in policy.py)")
print(f"   Our problem needs ≤ 150 steps (optimistic)")
print(f"   ✅ 500 should be enough!")

print(f"\n❓ So why are we hitting 500?")
print(f"   → Agents getting stuck in loops?")
print(f"   → Constraints blocking all valid actions?")
print(f"   → Decoder selecting same node repeatedly?")

📊 THEORETICAL STEP REQUIREMENTS

📐 Calculation:
   Customers to visit: 20
   Agents available: 4
   Optimal distribution: 5.0 customers per agent

🔢 Step Estimates:
   Best case (balanced): 24 steps
     = 20 customer visits + 4 returns to depot

   Realistic case (unbalanced): 28 steps
     = 20 customer visits + 8 intermediate depot returns

   Worst case (all back to depot between customers): 40 steps
     = each customer requires go + return = 40

⚠️  Environment Limit:
   max_steps = 500 (set in policy.py)
   Our problem needs ≤ 150 steps (optimistic)
   ✅ 500 should be enough!

❓ So why are we hitting 500?
   → Agents getting stuck in loops?
   → Constraints blocking all valid actions?
   → Decoder selecting same node repeatedly?


## 4️⃣ Debug Action Mask at Each Step

In [4]:
print("="*80)
print("🔍 DEBUGGING: Action Mask Evolution")
print("="*80)

# Use first instance from batch
td_single = td_reset[0:1].clone()
num_agents = td_single["current_node"].shape[-1]
num_nodes = td_single["action_mask"].shape[-1]

print(f"\n📦 Using batch instance 0")
print(f"   Nodes: {num_nodes} (depots + customers)")
print(f"   Agents: {num_agents}")

# Manually step through environment
step_history = []
max_debug_steps = 20  # Only trace first 20 steps

for step_num in range(max_debug_steps):
    # Get current state
    current_nodes = td_single["current_node"][0].tolist()
    visited = td_single["visited"][0].sum().item()
    mask = td_single["action_mask"][0]
    
    # Count available actions per agent
    available_per_agent = mask.sum(dim=-1).tolist()
    total_available = mask.sum().item()
    
    # Check if all agents at depot
    all_at_depot = all(node < num_agents for node in current_nodes)
    
    # Find agents with NO valid action
    stuck_agents = [i for i in range(num_agents) if available_per_agent[i] == 0]
    
    # Debug info
    status = "🟢 OK"
    if len(stuck_agents) > 0:
        status = f"🔴 STUCK: agents {stuck_agents}"
    elif all_at_depot and visited < num_nodes - num_agents:
        status = f"🟡 WARNING: all at depot, {num_nodes - num_agents - visited} unvisited"
    
    line = f"Step {step_num:3d}: visited={visited:2d}, available={total_available:2d}, per_agent={available_per_agent}, {status}"
    print(line)
    step_history.append({
        'step': step_num,
        'visited': visited,
        'available': total_available,
        'stuck': len(stuck_agents),
    })
    
    # If all agents stuck or all visited, stop
    if td_single["done"][0].item() or len(stuck_agents) == num_agents:
        print(f"\n✅ Episode ended at step {step_num}")
        break
    
    # Simulate action (greedy: pick first available)
    actions = []
    for agent_id in range(num_agents):
        valid_nodes = torch.where(mask[agent_id])[0]
        if len(valid_nodes) > 0:
            action = valid_nodes[0].item()
        else:
            action = current_nodes[agent_id]  # Stay if no valid
        actions.append(action)
    
    td_single["action"] = torch.tensor([actions], dtype=torch.int64)
    
    # Step environment
    td_single = env.step(td_single)["next"]

print(f"\n" + "="*80)
print(f"Summary: Episode progressed {step_num + 1} steps")
print(f"  - Customers visited: {td_single['visited'][0, num_agents:].sum().item()} / {num_nodes - num_agents}")
print(f"  - Episode done: {td_single['done'][0].item()}")

🔍 DEBUGGING: Action Mask Evolution

📦 Using batch instance 0
   Nodes: 24 (depots + customers)
   Agents: 4
Step   0: visited= 0, available=66, per_agent=[20, 20, 13, 13], 🟡 WARNING: all at depot, 20 unvisited
Step   1: visited= 2, available=16, per_agent=[2, 2, 6, 6], 🟢 OK
Step   2: visited= 6, available=26, per_agent=[1, 1, 12, 12], 🟡 WARNING: all at depot, 14 unvisited
Step   3: visited= 7, available= 6, per_agent=[1, 1, 2, 2], 🟢 OK
Step   4: visited= 7, available=20, per_agent=[1, 1, 9, 9], 🟡 WARNING: all at depot, 13 unvisited
Step   5: visited= 8, available=18, per_agent=[1, 1, 8, 8], 🟢 OK
Step   6: visited= 8, available=16, per_agent=[1, 1, 7, 7], 🟡 WARNING: all at depot, 12 unvisited
Step   7: visited= 9, available=14, per_agent=[1, 1, 6, 6], 🟢 OK
Step   8: visited= 9, available=12, per_agent=[1, 1, 5, 5], 🟡 WARNING: all at depot, 11 unvisited
Step   9: visited=10, available= 6, per_agent=[1, 1, 2, 2], 🟢 OK
Step  10: visited=10, available= 6, per_agent=[1, 1, 2, 2], 🟡 WARNING: 

## 5️⃣ Monitor Constraint Violations

In [7]:
print("="*80)
print("⚠️  CONSTRAINT VIOLATION ANALYSIS")
print("="*80)

# Reset again for clean analysis
td_single = td_reset[0:1].clone()

print(f"\nChecking which constraints block actions...\n")

max_steps_analysis = 30

for step_num in range(max_steps_analysis):
    current_nodes = td_single["current_node"][0].tolist()
    mask = td_single["action_mask"][0]
    
    # Get current state
    current_time = td_single["current_time"][0]
    trip_deadline = td_single["trip_deadline"][0]
    used_capacity = td_single["used_capacity"][0]
    agents_capacity = td_single["agents_capacity"][0]
    used_endurance = td_single["used_endurance"][0]
    agents_endurance = td_single["agents_endurance"][0]
    
    print(f"\n📍 Step {step_num}:")
    print(f"   Current nodes: {current_nodes}")
    print(f"   Current time: {[f'{t:.1f}' for t in current_time.tolist()]}")
    print(f"   Trip deadline: {[f'{d:.1f}' for d in trip_deadline.tolist()]}")
    print(f"   Capacity used: {[f'{c:.1f}' for c in used_capacity.tolist()]} / {agents_capacity[0].item():.1f}")
    print(f"   Endurance used: {[f'{e:.1f}' for e in used_endurance.tolist()]} / {agents_endurance[0].item():.1f}")
    
    # Check each agent
    for agent_id in range(num_agents):
        agent_mask = mask[agent_id]
        available = agent_mask.sum().item()
        
        if available == 0:
            print(f"   ❌ Agent {agent_id}: NO VALID ACTIONS (stuck!)")
            # Try to identify why
            remaining_cap = agents_capacity[agent_id].item() - used_capacity[agent_id].item()
            remaining_endurance = agents_endurance[agent_id].item() - used_endurance[agent_id].item()
            print(f"      Remaining capacity: {remaining_cap:.1f}")
            print(f"      Remaining endurance: {remaining_endurance:.1f}")
            print(f"      Current at: node {current_nodes[agent_id]} (depot={current_nodes[agent_id] < num_agents})")
        elif available <= 2:
            valid_nodes = torch.where(agent_mask)[0].tolist()
            print(f"   ⚠️  Agent {agent_id}: only {available} action(s) available: {valid_nodes}")
        else:
            print(f"   ✅ Agent {agent_id}: {available} actions available")
    
    # Greedy action
    actions = []
    for agent_id in range(num_agents):
        valid = torch.where(mask[agent_id])[0]
        action = valid[0].item() if len(valid) > 0 else current_nodes[agent_id]
        actions.append(action)
    
    td_single["action"] = torch.tensor([actions], dtype=torch.int64)
    td_single = env.step(td_single)["next"]
    
    if td_single["done"][0].item():
        print(f"\n✅ Episode ended at step {step_num}")
        break

print(f"\n" + "="*80)

⚠️  CONSTRAINT VIOLATION ANALYSIS

Checking which constraints block actions...


📍 Step 0:
   Current nodes: [0, 1, 2, 3]
   Current time: ['0.0', '0.0', '0.0', '0.0']
   Trip deadline: ['4009.3', '4009.3', '4009.3', '4009.3']
   Capacity used: ['0.0', '0.0', '0.0', '0.0'] / 200.0
   Endurance used: ['0.0', '0.0', '0.0', '0.0'] / 4009.3
   ✅ Agent 0: 20 actions available
   ✅ Agent 1: 20 actions available
   ✅ Agent 2: 13 actions available
   ✅ Agent 3: 13 actions available

📍 Step 1:
   Current nodes: [4, 4, 5, 5]
   Current time: ['1773.0', '1773.0', '206.7', '206.7']
   Trip deadline: ['5373.0', '5373.0', '3806.7', '3806.7']
   Capacity used: ['0.6', '0.6', '0.6', '0.6'] / 200.0
   Endurance used: ['833.3', '833.3', '206.7', '206.7'] / 4009.3
   ⚠️  Agent 0: only 2 action(s) available: [0, 12]
   ⚠️  Agent 1: only 2 action(s) available: [1, 12]
   ✅ Agent 2: 6 actions available
   ✅ Agent 3: 6 actions available

📍 Step 2:
   Current nodes: [0, 1, 2, 3]
   Current time: ['2606.3', '2

## 6️⃣ Identify Maximum Steps Bottleneck

In [6]:
print("="*80)
print("🎯 ROOT CAUSE ANALYSIS")
print("="*80)

# Run full episode and count steps
td_full = td_reset.clone()
step_counts = []

for batch_idx in range(td_full.batch_size[0]):
    td_single = td_full[batch_idx:batch_idx+1].clone()
    
    step = 0
    max_steps = 500
    
    while step < max_steps:
        mask = td_single["action_mask"]
        
        # Greedy action selection
        actions = []
        for agent_id in range(num_agents):
            agent_mask = mask[0, agent_id]
            valid = torch.where(agent_mask)[0]
            action = valid[0].item() if len(valid) > 0 else td_single["current_node"][0, agent_id].item()
            actions.append(action)
        
        td_single["action"] = torch.tensor([actions], dtype=torch.int64)
        td_single = env.step(td_single)["next"]
        step += 1
        
        if td_single["done"][0].item():
            break
    
    visited = td_single["visited"][0, num_agents:].sum().item()
    step_counts.append((step, visited, num_nodes - num_agents))
    
    status = "✅ SUCCESS" if step < max_steps else "❌ MAX STEPS HIT"
    print(f"Instance {batch_idx}: {status} | Steps: {step:3d} | Visited: {visited:2d}/{num_nodes - num_agents}")

print(f"\n" + "="*80)
print(f"SUMMARY:")
print(f"="*80)

step_list = [s[0] for s in step_counts]
avg_steps = np.mean(step_list)
max_steps_taken = np.max(step_list)
min_steps_taken = np.min(step_list)

print(f"\n📊 Statistics:")
print(f"   Average steps: {avg_steps:.1f}")
print(f"   Max steps taken: {max_steps_taken}")
print(f"   Min steps taken: {min_steps_taken}")

print(f"\n🔍 Analysis:")
if avg_steps > 200:
    print(f"   ⚠️  Average steps ({avg_steps:.0f}) is HIGH!")
    print(f"   → Constraints are blocking many actions")
    print(f"   → Agents spending time in loops")
    print(f"   → Check: time windows, deadlines, endurance limits")
else:
    print(f"   ✅ Average steps ({avg_steps:.0f}) is reasonable")

if max_steps_taken >= 500:
    print(f"\n   ❌ {sum(1 for s in step_counts if s[0] >= 500)} instances hit max_steps=500")
    print(f"   → These instances are INCOMPLETE")
    unvisited = [s[1] for s in step_counts if s[0] >= 500]
    print(f"   → Average unvisited customers: {np.mean([s[2] - s[1] for s in step_counts if s[0] >= 500]):.1f}")
else:
    print(f"\n   ✅ All instances completed before max_steps")

print(f"\n💡 CONCLUSION:")
print(f"   If avg_steps ≈ 20-50: ✅ Problem is solvable")
print(f"   If avg_steps > 150: ⚠️  Constraints too tight")
print(f"   If hitting 500: ❌ Stuck loop issue detected")

🎯 ROOT CAUSE ANALYSIS
Instance 0: ✅ SUCCESS | Steps:  12 | Visited:  7/20
Instance 1: ✅ SUCCESS | Steps:  12 | Visited:  8/20
Instance 2: ✅ SUCCESS | Steps:   9 | Visited:  8/20
Instance 3: ✅ SUCCESS | Steps:   9 | Visited:  7/20

SUMMARY:

📊 Statistics:
   Average steps: 10.5
   Max steps taken: 12
   Min steps taken: 9

🔍 Analysis:
   ✅ Average steps (10) is reasonable

   ✅ All instances completed before max_steps

💡 CONCLUSION:
   If avg_steps ≈ 20-50: ✅ Problem is solvable
   If avg_steps > 150: ⚠️  Constraints too tight
   If hitting 500: ❌ Stuck loop issue detected


## Next Steps

Based on the output above:

- **If hitting max_steps = 500**: Check constraint violations in Step 5️⃣. The stuck_detected logic should help.
- **If low average steps (< 100)**: Problem is normal, increase max_steps if needed
- **If high average steps (> 200)**: Check constraint tightness - may need to adjust problem parameters